In [ ]:
import numpy as np
import plotly.express as px
import polars as pl
from IPython.display import display  # noqa: A004


def calibration(df: pl.DataFrame, name: str) -> None:
    calibration_ratio = df.select(pl.mean("y_prob") / pl.mean("y_true")).item()
    labels = [str(o) for o in range(10)]
    calibration_curve = (
        df.with_columns(pl.col("y_prob").qcut(10, labels=labels).alias("bin"))
        .group_by("bin")
        .agg(
            pl.len(),
            pl.mean("y_true").alias("pct_true"),
            pl.mean("y_prob").alias("pct_prob"),
        )
        .sort("bin")
    )
    ece = (
        calibration_curve.with_columns(
            ((pl.col("pct_true") - pl.col("pct_prob")).abs() * pl.col("len")).alias(
                "weighted_error"
            )
        )
        .select(pl.sum("weighted_error") / pl.sum("len"))
        .item()
    )
    title: str = f"{name} - ECE: {ece:.2f} - calibration_ratio: {calibration_ratio:.2f}"
    f = px.line(calibration_curve, "pct_prob", "pct_true", title=title, markers=True)
    f.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1, line={"color": "gray", "dash": "dash"}
    )
    display(f)


n: int = 10_000
df = pl.DataFrame(
    {
        "y_prob": np.random.rand(n),
        "y_true": np.random.randint(0, 2, n),
    }
)
calibration(df, "random")
df = pl.DataFrame(
    {
        "y_prob": np.linspace(0.0001, 0.9999, n),
        "y_true": np.sort(np.random.randint(0, 2, n)),
    }
)
calibration(df, "perfect discrimination")
y_prob = np.random.rand(n)
y_true = np.random.binomial(1, y_prob)
df = pl.DataFrame({"y_prob": y_prob, "y_true": y_true})
calibration(df, "perfect calibration")